# 06 · Screening 7,206 CoRE MOF 2019 structures, and fine-tuning the 77 K model
Reproduces report sections 6.6–6.7, in the order the project ran them:
1. Featurise all 7,206 accessible MOFs (fast Zeo++ settings) and predict DOE capacity (`screen_featurise.py`, `screen_predict.py`).
2. GCMC on the top 10 and Widom on the top 300, then refine (`screen_predict.py refine`).
3. 77 K GCMC on 150 real MOFs and fine-tune the 77 K model, then re-screen (`finetune_77.py`).
4. GCMC on the new top 20 and the bottom 5.

**Runtime: about 10–14 hours on 4 cores: run it across two sessions** (save, attach output, rerun). **Attach:** the `mofh2_pipeline` dataset, and stage 05 output (or keep `USE_REFERENCE_UPSTREAM = True`).

### Settings
* `USE_REFERENCE_UPSTREAM`: fill results this notebook needs from earlier stages with the original project's runs (packaged in `reference_runs.tgz`) when you have not attached your own earlier notebook outputs. Lets every notebook run on its own.
* `RECOMPUTE_THIS_STAGE`: re-run this notebook's own simulations. Set `False` to only redo the analysis on the reference runs.
* `MAX_HOURS`: stop launching new simulations after this many hours (Kaggle sessions end at 12 h). Every queue is resumable: save the output, attach it to a new session, run again.

In [ ]:
import os, glob, subprocess
PKG = os.path.dirname(glob.glob('/kaggle/input/**/mofh2_pipeline/setup_env.sh', recursive=True)[0])
os.environ['WORK'] = WORK = '/kaggle/working/mofh2'
os.environ['NPROC'] = str(os.cpu_count())
USE_REFERENCE_UPSTREAM = True
RECOMPUTE_THIS_STAGE = True
MAX_HOURS = 11.0
STAGE_QUEUES = ['06a_screen_top10_widom300', '06b_ft77_labels', '06c_top20_bottom5']
print('package:', PKG, '| cores:', os.cpu_count())

### Setup: build RASPA2 and Zeo++, install packages (about 5–8 minutes)

In [ ]:
!bash $PKG/setup_env.sh

### Restore results from earlier stages

In [ ]:
%cd $WORK
!python stage_io.py restore {'--reference' if USE_REFERENCE_UPSTREAM else ''}
if RECOMPUTE_THIS_STAGE:
    for q in STAGE_QUEUES:
        !python stage_io.py forget queues/{q}.txt

In [ ]:
if not os.path.isdir(f'{WORK}/coredl/MOFSimplify/CoRE2019'):
    !mkdir -p coredl && cd coredl && git clone -q --depth 1 --filter=blob:none --no-checkout https://github.com/hjkgrp/MOFSimplify && cd MOFSimplify && git sparse-checkout set --no-cone 'CoRE2019/*' && git checkout -q HEAD
if not os.path.exists('core_res.csv') or not os.path.exists('core_meta.csv'):
    !python core_meta.py coredl/MOFSimplify/CoRE2019 core_meta.csv && python core_res.py
!python featurise_core.py && python featurise_real.py

### 1 · Featurise and score all 7,206 MOFs (about 20–40 minutes)

In [ ]:
!python screen_featurise.py && python screen_predict.py

### Run simulations: `06a_screen_top10_widom300` (top-10 GCMC + top-300 Widom)

In [ ]:
!bash rq_any.sh queues/06a_screen_top10_widom300.txt {MAX_HOURS}

In [ ]:
!python screen_predict.py refine

### Run simulations: `06b_ft77_labels` (150 real MOFs at 77 K, 2 and 100 bar; 30 test MOFs at 2 bar)

In [ ]:
!bash rq_any.sh queues/06b_ft77_labels.txt {MAX_HOURS}

### 3 · Fine-tune the 77 K model, then re-screen

In [ ]:
!python av_feature.py ft && python finetune_77.py

### Run simulations: `06c_top20_bottom5` (new top 20 and bottom 5)

In [ ]:
!bash rq_any.sh queues/06c_top20_bottom5.txt {MAX_HOURS}

### 4 · Top-20 and bottom-5 check against GCMC

In [ ]:
import sys, numpy as np, pandas as pd; sys.path.insert(0, WORK)
from extract import parse_data
def gl(d):
    f = glob.glob(f'{d}/Output/System_0/*.data')
    if not f: return np.nan
    r = parse_data(f[0]); return r['gL'] if r.get('finished') else np.nan
d = pd.read_csv('screen_core2019_ft77.csv')
for part, sub in [('top20', d.head(20)), ('bottom5', d.tail(5))]:
    sub = sub.copy(); sub['g100'] = [gl(f'runs_m2/{x}_100') for x in sub.raspa]; sub['g160'] = [gl(f'runs_160/{x}_5') for x in sub.raspa]
    sub['doe_gcmc'] = sub.g100 - sub.g160
    print(part, 'GCMC DOE capacity: mean %.1f, range %.1f-%.1f g/L' % (sub.doe_gcmc.mean(), sub.doe_gcmc.min(), sub.doe_gcmc.max()))
    sub.to_csv(f'screen_{part}_ft_gcmc_repro.csv', index=False)
print('Original project: top 20 all >= 50 g/L (mean 52.6, max 56.8, OJOSEN); bottom 5 3.9-6.8 g/L')

### Compare with the original project and save outputs
The comparison prints mean and maximum differences between your reproduced numbers and the original ones. GCMC is stochastic, so expect small differences (typically under 1 g/L at 77 K/100 bar).

In [ ]:
!python compare_reference.py $PKG finetune_77_results.csv screen_core2019_ft77.csv

In [ ]:
!python stage_io.py save stage06_outputs
# Attach /kaggle/working/stage06_outputs.tgz (via 'Save Version' output) as input to later notebooks.